# 🐄 CattlePrint - Deep Metric Learning Training Pipeline (Google Colab)

This notebook trains an **ArcFace (Additive Angular Margin Loss)** neural network on bovine muzzle prints for biometric identification.

### Pipeline Overview:
1. **Environment Setup & GPU Check** (PyTorch, Torchvision, ONNX, OpenCV)
2. **Open Dataset Download & Preprocessing** (Zenodo, Hugging Face, Mendeley)
3. **Muzzle ROI Detection & CLAHE Texture Enhancement**
4. **ArcFace Training Loop with Angular Margin** ($m=0.50, s=64.0$)
5. **Biometric Evaluation** (FAR, FRR, EER, ROC-AUC, Rank-1 Identification)
6. **Export Trained Weights & ONNX Model**

In [ ]:
# Step 1: Install Dependencies & Check GPU
!nvidia-smi
!pip install -q torch torchvision opencv-python onnx scikit-learn pillow

In [ ]:
# Step 2: Clone or Setup CattlePrint Repository
import torch, os
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using compute device: {device}")

In [ ]:
# Step 3: Dataset Ingestion & Preprocessing
# Public datasets referenced in CattlePrint:
# 1. https://zenodo.org/records/6324361
# 2. https://huggingface.co/datasets/MingShuo666/CattleMuzzle
# 3. https://data.mendeley.com/datasets/w7j9wb9gt2/1
# 4. https://zenodo.org/records/10535934
# 5. https://huggingface.co/datasets/casperhuggingface/gaaibastu-muzzles

from pathlib import Path
from ml_engine.datasets.downloader import setup_local_benchmark_dataset, print_catalog

print_catalog()
# Prepares dataset directory
data_path = Path('./cattle_dataset')
setup_local_benchmark_dataset(data_path, num_cattle=20, samples_per_cattle=5)
print(f"Dataset prepared at {data_path.resolve()}")

In [ ]:
# Step 4: Run ArcFace Deep Metric Training
from ml_engine.train import train_cattle_model

trained_model, metrics = train_cattle_model(
    data_dir=data_path,
    num_epochs=15,
    batch_size=8,
    learning_rate=3e-4,
    device=str(device)
)

print("\n--- FINAL BIOMETRIC PERFORMANCE ---")
for metric_name, val in metrics.items():
    print(f"{metric_name:25s}: {val}")

In [ ]:
# Step 5: Download Exported ONNX & PyTorch Checkpoints
from google.colab import files
import glob

onnx_files = glob.glob('ml_engine/exports/*.onnx')
pt_files = glob.glob('ml_engine/weights/*.pt')

print("Found files to download:", onnx_files + pt_files)
for f in onnx_files + pt_files:
    files.download(f)